# YOLO11n-seg tanpa pretrained weights

Notebook pembanding ini memakai Panoptic Indoor Segmentation v5 dan pengaturan yang sama dengan baseline: 50 epoch, image size 640, batch 16. Model dimulai dari bobot acak dengan konfigurasi `yolo11n-seg.yaml`, bukan file bobot `.pt`.

Di Kaggle, aktifkan GPU. Tambahkan Kaggle Secret bernama `ROBOFLOW_API_KEY` berisi API key Roboflow (tanpa tanda kutip).

In [1]:
!pip install -q ultralytics roboflow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
from pathlib import Path
from roboflow import Roboflow
from ultralytics import YOLO
from kaggle_secrets import UserSecretsClient
import yaml, torch, shutil

assert torch.cuda.is_available(), 'Aktifkan GPU di pengaturan Kaggle.'
print('GPU:', torch.cuda.get_device_name(0))
api_key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key)
project = rf.workspace('panopticindoor').project('panoptic-indoor-segmentation')
dataset = project.version(5).download('yolov11')
data_yaml = Path(dataset.location) / 'data.yaml'
with open(data_yaml, 'r') as f:
    data_cfg = yaml.safe_load(f)
print('Dataset:', dataset.location)
print('Splits:', {k: data_cfg.get(k) for k in ['train', 'val', 'test']})

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
GPU: Tesla T4
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to panoptic-indoor-segmentation-5 in yolov11:: 100%|██████████| 1856/1856 [00:00<00:00, 8100.22it/s]

Dataset: /kaggle/working/panoptic-indoor-segmentation-5
Splits: {'train': '../train/images', 'val': '../valid/images', 'test': '../test/images'}


## Training dari bobot acak
Baris `YOLO('yolo11n-seg.yaml')` membangun arsitektur tanpa memuat pretrained weights. Jangan menggantinya dengan file `.pt`.

In [3]:
model = YOLO('yolo11n-seg.yaml')  # arsitektur sama, bobot awal acak
train_result = model.train(
    data=str(data_yaml),
    task='segment',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project='/kaggle/working/runs',
    name='yolo11n_seg_no_pretrained',
    exist_ok=True,
    plots=True,
    seed=42
)
run_dir = Path(model.trainer.save_dir)
best_weights = run_dir / 'weights' / 'best.pt'
print('Training folder:', run_dir)
print('Best weights:', best_weights)
print('Training log:', run_dir / 'results.csv')

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/panoptic-indoor-segmentation-5/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_seg_no_pretrained,

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


       2/50      4.41G      3.481      8.239      5.473      4.214          0        353        640: 100% ━━━━━━━━━━━━ 25/25 3.9it/s 6.3s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.6it/s 1.7s0.3s
                   all        166       2021          0          0          0          0          0          0          0          0

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/50      4.41G      3.491      6.864      5.462      4.184          0        336        640: 100% ━━━━━━━━━━━━ 25/25 3.9it/s 6.4s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.5it/s 1.7s0.4s
                   all        166       2021          0          0          0          0          0          0          0          0

  

## Evaluasi pada test set
Notebook menggunakan split `test`. Jika split itu tidak ada di konfigurasi dataset, sel akan berhenti agar hasil val tidak keliru disebut hasil test.

In [4]:
assert data_cfg.get('test'), 'Split test tidak ditemukan di data.yaml.'
best_model = YOLO(str(best_weights))
test_metrics = best_model.val(data=str(data_yaml), split='test', imgsz=640, device=0, plots=True)
test_report = (
    f"Evaluated split: test\n"
    f"Box mAP50-95: {test_metrics.box.map:.6f}\n"
    f"Mask mAP50: {test_metrics.seg.map50:.6f}\n"
    f"Mask mAP50-95: {test_metrics.seg.map:.6f}\n"
)
print(test_report)
(run_dir / 'test_metrics.txt').write_text(test_report)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n-seg summary (fused): 113 layers, 2,860,198 parameters, 0 gradients, 9.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1586.6±1225.0 MB/s, size: 69.3 KB)
val: Scanning /kaggle/working/panoptic-indoor-segmentation-5/test/labels... 359 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 359/359 775.6it/s 0.5s0.1s
val: /kaggle/working/panoptic-indoor-segmentation-5/test/images/002_jpg.rf.941b44d4633817fd833de04127e2f3a3.jpg: 1 duplicate labels removed
val: /kaggle/working/panoptic-indoor-segmentation-5/test/images/236_jpeg.rf.ce43674993187cb554fcbd2c85146baf.jpg: 2 duplicate labels removed
val: New cache created: /kaggle/working/panoptic-indoor-segmentation-5/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 4.2it/s 5.5s0.2s
                   all        359       

90

## Siapkan file untuk diunduh
Berkas yang disiapkan: `best.pt`, `results.csv`, dan `test_metrics.txt`. Unduh dari panel Output Kaggle setelah notebook selesai dijalankan.

In [5]:
export_dir = Path('/kaggle/working/yolo11n_no_pretrained')
export_dir.mkdir(parents=True, exist_ok=True)
for filename in ['weights/best.pt', 'results.csv', 'test_metrics.txt']:
    source = run_dir / filename
    if source.exists():
        shutil.copy2(source, export_dir / Path(filename).name)
    else:
        print('Tidak ditemukan:', source)
print('File siap diunduh:', sorted(p.name for p in export_dir.iterdir()))

File siap diunduh: ['best.pt', 'results.csv', 'test_metrics.txt']
